# SSM vs Jamba vs Transformer — Colab Runner (A100, resumable)

This notebook trains and benchmarks three parameter-matched ~125M models (Transformer,
Mamba-2, Jamba) and is **resumable**: if Colab disconnects (12h cap / idle), just re-run
the notebook top-to-bottom. Finished runs are skipped (`DONE` marker) and in-progress runs
resume from their last checkpoint (exact step / LR / data position / RNG).

Everything (code, checkpoints, tokenized data, results) lives on Google Drive so a fresh
runtime + re-run is a complete recovery path.

**Order:** mount Drive → sync repo → install (with kernel-or-fallback) → one-time data prep
→ check params → train all archs → synthetic sweep → efficiency sweep → aggregate + plots.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Sync the latest code onto Drive (safe, preserves runs/ + data)
Downloads the latest code from GitHub and overwrites only code files — your `runs/`
checkpoints, `results/`, and the tokenized data are left untouched. Re-run this every
session so you always have the current fixes. (Repo must be public; if private, set a token.)

In [ ]:
import os, sys, io, tarfile, urllib.request
GITHUB_USER = 'amirhossein-yousefi'
GITHUB_REPO = 'SSM-Project'
BRANCH = 'main'
PROJECT_DIR = '/content/drive/MyDrive/ssm'
DATA_DIR = '/content/drive/MyDrive/ssm_data/fineweb_edu_gpt2'
WHEELS = '/content/drive/MyDrive/ssm_wheels'

# Never overwrite these local dirs (checkpoints / results / caches live here)
PROTECT = ('runs/', 'results/', 'data_cache/', 'wheels/')
os.makedirs(PROJECT_DIR, exist_ok=True)

url = f'https://github.com/{GITHUB_USER}/{GITHUB_REPO}/archive/refs/heads/{BRANCH}.tar.gz'
print('downloading', url)
blob = urllib.request.urlopen(url, timeout=60).read()
tf = tarfile.open(fileobj=io.BytesIO(blob))
prefix = tf.getnames()[0].split('/')[0] + '/'
n = 0
for m in tf.getmembers():
    rel = m.name[len(prefix):] if m.name.startswith(prefix) else m.name
    if not rel or any(rel.startswith(p) for p in PROTECT):
        continue
    dest = os.path.join(PROJECT_DIR, rel)
    if m.isdir():
        os.makedirs(dest, exist_ok=True)
    elif m.isfile():
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        with open(dest, 'wb') as f:
            f.write(tf.extractfile(m).read())
        n += 1
print(f'updated {n} code files into {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
SRC = os.path.join(PROJECT_DIR, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)
os.environ['PYTHONPATH'] = SRC + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('cwd:', os.getcwd(), '| ssm_bench importable from', SRC)


## 3. Install deps (with kernel-or-fallback)
Do **not** reinstall torch (Colab provides it). Try to build the fast Mamba CUDA kernels;
if that fails, the harness automatically uses HuggingFace's correct (slower) torch path.
Successful builds are cached to Drive as wheels so future sessions install in seconds.

In [ ]:
import os, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-r', 'requirements.txt'], check=True)
# editable install so `ssm_bench` is importable everywhere (best-effort; PYTHONPATH is the fallback)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', '-e', '.'], check=False)
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'ninja', 'packaging'], check=False)

os.makedirs(WHEELS, exist_ok=True)
USE_MAMBA_KERNELS = False
# 1) try cached wheels first (fast)
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index',
                    f'--find-links={WHEELS}', 'mamba-ssm', 'causal-conv1d'], check=True)
    import mamba_ssm, causal_conv1d  # noqa
    USE_MAMBA_KERNELS = True
    print('Installed Mamba kernels from cached wheels.')
except Exception:
    # 2) build from source (slow, may fail) then cache wheels
    try:
        env = dict(os.environ, MAX_JOBS='4')
        subprocess.run([sys.executable, '-m', 'pip', 'install',
                        'causal-conv1d>=1.4.0', 'mamba-ssm>=2.2.2', '--no-build-isolation'],
                       check=True, env=env, timeout=1800)
        import mamba_ssm, causal_conv1d  # noqa
        USE_MAMBA_KERNELS = True
        subprocess.run([sys.executable, '-m', 'pip', 'wheel', 'mamba-ssm', 'causal-conv1d',
                        '-w', WHEELS], check=False)
        print('Built + cached Mamba kernels.')
    except Exception as e:
        print(f'Kernel build failed -> using eager fallback (correct, slower). {e}')
print('USE_MAMBA_KERNELS =', USE_MAMBA_KERNELS)


## 4. One-time data prep (FineWeb-Edu → uint16 shards on Drive)
Runs only if the tokenized cache is missing. ~1.5B tokens, a few GB on Drive; takes a while
the first time, then every session reuses it.

In [ ]:
import os, subprocess, sys
if not os.path.exists(os.path.join(DATA_DIR, 'manifest.json')):
    subprocess.run([sys.executable, '-m', 'ssm_bench.data.prepare_fineweb', '--out', DATA_DIR],
                   check=True)
else:
    print('Tokenized data already present at', DATA_DIR)


## 5. (Optional) Verify the three models are parameter-matched (~125M ± 5%)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'ssm_bench.models.param_utils', '--check'], check=False)


## 6. Train all architectures (resumable)
Skips any arch with a `DONE` marker; otherwise resumes from its last checkpoint. Re-run this
cell after any disconnect to continue. Tune `TOTAL_STEPS` / batch to your time budget.

In [ ]:
import os, subprocess, sys
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
ARCHS = ['transformer', 'mamba', 'jamba']
SEED = 1337
TOTAL_STEPS = 6000
# Per-arm micro_batch/grad_accum; micro_batch * grad_accum == 512 for ALL arms, so the
# effective batch / optimization is identical (accumulation is a pure memory knob).
# Jamba uses the fast Mamba CUDA kernel with autocast OFF (fp32) -> avoids the kernel/bf16
# dtype bug AND the slow torch path; fp32 logits use more memory so a smaller micro-batch.
BATCH = {'transformer': (16, 32), 'mamba': (16, 32), 'jamba': (8, 64)}
RUNS_DIR = os.path.join(PROJECT_DIR, 'runs')
for arch in ARCHS:
    run = os.path.join(RUNS_DIR, f'{arch}_seed{SEED}')
    if os.path.exists(os.path.join(run, 'DONE')):
        print('SKIP', arch, '(DONE)'); continue
    mb, ga = BATCH[arch]
    cmd = [sys.executable, 'scripts/train.py',
           '--arch', arch, '--seed', str(SEED),
           '--data_dir', DATA_DIR, '--output_dir', run,
           '--total_steps', str(TOTAL_STEPS),
           '--micro_batch_size', str(mb), '--grad_accum', str(ga),
           '--stage_dir', '/content']
    if arch == 'jamba':
        cmd += ['--force_kernels', '--no_autocast']  # fast kernel path (fp32, no autocast)
    print(f'=== TRAIN {arch}  (micro_batch={mb}, grad_accum={ga}) ===')
    subprocess.run(cmd, check=True)   # streams logs live (do NOT capture_output)


## 7. Strided-window perplexity for each trained run

In [ ]:
import os, subprocess, sys
SEED = 1337
for arch in ['transformer', 'mamba', 'jamba']:
    run = os.path.join(PROJECT_DIR, 'runs', f'{arch}_seed{SEED}')
    if os.path.exists(os.path.join(run, 'DONE')):
        subprocess.run([sys.executable, '-m', 'ssm_bench.eval.lm_quality',
                        '--run_dir', run, '--data_dir', DATA_DIR], check=False)


## 8. Synthetic mechanistic-task sweep (MQAR / induction / selective-copy)

In [ ]:
import subprocess
subprocess.run(['bash', 'scripts/run_synthetics.sh'], check=True)


## 9. Efficiency benchmark (throughput / memory / prefill / decode)

In [ ]:
import subprocess
subprocess.run(['bash', 'scripts/run_efficiency.sh'], check=True)


## 10. Aggregate + plot, then display the figures

In [ ]:
import subprocess, sys, glob
from IPython.display import Image, display
subprocess.run([sys.executable, 'scripts/aggregate.py'], check=True)
subprocess.run([sys.executable, 'scripts/plots.py'], check=True)
for p in sorted(glob.glob('results/figures/*.png')):
    print(p); display(Image(p))


## 11. Copy results back to your machine
Everything you need to commit lives in `results/summaries/*.csv` and `results/figures/*.png`
on Drive (under `MyDrive/ssm`). Download those folders and drop them into your local repo's
`results/`, then commit.

**Keep-alive note:** checkpoints save every ~250 steps, and a SIGTERM handler flushes
`last.pt` on kill, so a disconnect loses at most a few minutes. The 12h cap is handled purely
by re-running this notebook (it resumes). No keep-alive hack is required.